# Lung Cancer CT Classification
Pretrained EfficientNet fine-tuning

In [1]:

!pip install -q torch torchvision


In [2]:
import os
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [6]:
!unzip /content/lung-cancer-dataset.zip -d /content/

Archive:  /content/lung-cancer-dataset.zip
   creating: /content/lung-cancer-dataset/
   creating: /content/lung-cancer-dataset/test/
   creating: /content/lung-cancer-dataset/test/adenocarcinoma/
  inflating: /content/lung-cancer-dataset/test/adenocarcinoma/000108 (3).png  
  inflating: /content/lung-cancer-dataset/test/adenocarcinoma/000109 (2).png  
  inflating: /content/lung-cancer-dataset/test/adenocarcinoma/000109 (4).png  
  inflating: /content/lung-cancer-dataset/test/adenocarcinoma/000109 (5).png  
  inflating: /content/lung-cancer-dataset/test/adenocarcinoma/000112 (2).png  
  inflating: /content/lung-cancer-dataset/test/adenocarcinoma/000113 (7).png  
  inflating: /content/lung-cancer-dataset/test/adenocarcinoma/000114 (5).png  
  inflating: /content/lung-cancer-dataset/test/adenocarcinoma/000114.png  
  inflating: /content/lung-cancer-dataset/test/adenocarcinoma/000115 (4).png  
  inflating: /content/lung-cancer-dataset/test/adenocarcinoma/000115 (8).png  
  inflating: /con

In [7]:
base_dir = "/content/lung-cancer-dataset"

train_dir = os.path.join(base_dir, "train")
val_dir   = os.path.join(base_dir, "valid")
test_dir  = os.path.join(base_dir, "test")

print("Train:", os.listdir(train_dir))
print("Test:", os.listdir(test_dir))
print("Valid:", os.listdir(val_dir))

Train: ['squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa', 'normal', 'adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib', 'large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa']
Test: ['squamous.cell.carcinoma', 'adenocarcinoma', 'normal', 'large.cell.carcinoma']
Valid: ['squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa', 'normal', 'adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib', 'large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa']


In [8]:
print(os.listdir("/content/lung-cancer-dataset"))


['train', 'valid', 'test']


In [9]:
IMG_SIZE = 224

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.9, 1.1)
    ),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [10]:
import os
import shutil

base_dir = "/content/lung-cancer-dataset"

# Remove .ipynb_checkpoints directories if they exist
for root, dirs, files in os.walk(base_dir):
    for dir_name in dirs:
        if '.ipynb_checkpoints' in dir_name:
            checkpoint_path = os.path.join(root, dir_name)
            print(f"Removing: {checkpoint_path}")
            shutil.rmtree(checkpoint_path)

In [11]:
train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
val_ds   = datasets.ImageFolder(val_dir, transform=val_tfms)
test_ds  = datasets.ImageFolder(test_dir, transform=val_tfms)

class_names = train_ds.classes
num_classes = len(class_names)

print("Classes:", class_names)


Classes: ['adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib', 'large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa', 'normal', 'squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa']


In [12]:
BATCH_SIZE = 16

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


In [13]:
targets = [label for _, label in train_ds.samples]
class_counts = np.bincount(targets)
weights = 1. / torch.tensor(class_counts, dtype=torch.float)

class_weights = weights.to(device)
print("Class weights:", class_weights)


Class weights: tensor([0.0051, 0.0087, 0.0068, 0.0065], device='cuda:0')


In [14]:
weights = models.EfficientNet_B3_Weights.IMAGENET1K_V1
model = models.efficientnet_b3(weights=weights)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    num_classes
)

model = model.to(device)


Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:01<00:00, 45.0MB/s]


In [15]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)


In [16]:
EPOCHS = 25

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0
    correct = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()

    acc = correct / len(val_ds)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss {train_loss:.3f} | Val Acc {acc:.3f}")


Epoch 1/25 | Train Loss 48.881 | Val Acc 0.514
Epoch 2/25 | Train Loss 37.208 | Val Acc 0.653
Epoch 3/25 | Train Loss 28.556 | Val Acc 0.694
Epoch 4/25 | Train Loss 20.080 | Val Acc 0.722
Epoch 5/25 | Train Loss 14.551 | Val Acc 0.819
Epoch 6/25 | Train Loss 12.785 | Val Acc 0.833
Epoch 7/25 | Train Loss 8.978 | Val Acc 0.861
Epoch 8/25 | Train Loss 7.059 | Val Acc 0.903
Epoch 9/25 | Train Loss 6.689 | Val Acc 0.833
Epoch 10/25 | Train Loss 6.169 | Val Acc 0.847
Epoch 11/25 | Train Loss 5.649 | Val Acc 0.861
Epoch 12/25 | Train Loss 5.799 | Val Acc 0.875
Epoch 13/25 | Train Loss 4.166 | Val Acc 0.875
Epoch 14/25 | Train Loss 2.880 | Val Acc 0.875
Epoch 15/25 | Train Loss 2.837 | Val Acc 0.875
Epoch 16/25 | Train Loss 3.003 | Val Acc 0.875
Epoch 17/25 | Train Loss 2.825 | Val Acc 0.875
Epoch 18/25 | Train Loss 2.448 | Val Acc 0.889
Epoch 19/25 | Train Loss 3.492 | Val Acc 0.875
Epoch 20/25 | Train Loss 2.275 | Val Acc 0.875
Epoch 21/25 | Train Loss 2.277 | Val Acc 0.875
Epoch 22/25 | Tr

In [17]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        outputs = model(imgs.to(device))
        preds = outputs.argmax(1).cpu()

        y_true.extend(labels)
        y_pred.extend(preds)

print(classification_report(y_true, y_pred, target_names=class_names))
print(confusion_matrix(y_true, y_pred))


                                                  precision    recall  f1-score   support

      adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib       1.00      0.73      0.85       120
   large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa       0.77      0.96      0.85        51
                                          normal       1.00      0.98      0.99        54
squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa       0.81      0.99      0.89        90

                                        accuracy                           0.89       315
                                       macro avg       0.89      0.92      0.89       315
                                    weighted avg       0.91      0.89      0.88       315

[[88 13  0 19]
 [ 0 49  0  2]
 [ 0  1 53  0]
 [ 0  1  0 89]]


In [19]:
model.eval()
indices = random.sample(range(len(test_ds)), 12)

for idx in indices:
    img, label = test_ds[idx]
    output = model(img.unsqueeze(0).to(device))
    pred = output.argmax(1).item()

    print(f"Actual: {class_names[label]} | Predicted: {class_names[pred]}")


Actual: squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa | Predicted: squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa
Actual: adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib | Predicted: adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib
Actual: large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa | Predicted: large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa
Actual: normal | Predicted: normal
Actual: large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa | Predicted: large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa
Actual: large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa | Predicted: large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa
Actual: normal | Predicted: normal
Actual: large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa | Predicted: large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa
Actual: squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa | Predicted: squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa
Actual: adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib | Predicted: adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib
Actual: adenocarcinoma_lef

In [20]:
torch.save({
    "model_state": model.state_dict(),
    "class_names": class_names
}, "/content/lung_cancer_efficientnet_b3_final3.pth")

print("Model saved ✅")


Model saved ✅


In [21]:
checkpoint = torch.load("/content/lung_cancer_efficientnet_b3_final3.pth", map_location=device)

weights = models.EfficientNet_B3_Weights.IMAGENET1K_V1
model = models.efficientnet_b3(weights=weights)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(checkpoint["class_names"]))

model.load_state_dict(checkpoint["model_state"])
model = model.to(device)
model.eval()

class_names = checkpoint["class_names"]
print("Model loaded ✅")


Model loaded ✅


In [31]:
from google.colab import files
from PIL import Image
import torch

# Upload image
uploaded = files.upload()

# Get uploaded image path
image_path = list(uploaded.keys())[0]

# Load and preprocess
image = Image.open(image_path).convert("RGB")
tensor = val_tfms(image).unsqueeze(0).to(device)

# Predict
model.eval()
with torch.no_grad():
    output = model(tensor)
    probs = torch.softmax(output, dim=1)[0]
    pred = probs.argmax().item()

print("Prediction:", class_names[pred])
print("Confidence:", round(probs[pred].item() * 100, 2), "%")


Saving largecell.jpg to largecell (2).jpg
Prediction: large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa
Confidence: 99.8 %
